# Retinal Disease Classification from OCT ImagesTraining script for the MobileNet V3 fine-tuned model used in the viewer.Reproduces `ml_models/eye_disease/trained_model.keras`.**Architecture:** MobileNet V3 Large (ImageNet weights) + Dense(4, softmax)**Input:** 224 x 224 x 3 (RGB)**Classes:** CNV, DME, DRUSEN, NORMAL**Dataset:** Kermany OCT — https://www.kaggle.com/datasets/paultimothymooney/kermany2018 (84,495 images)**Reference code:** `Human_Eye_Disease_Prediction/Training_model.ipynb`---

## 1. Setup

In [ ]:
import tensorflow as tffrom tensorflow.keras.utils import image_dataset_from_directoryimport matplotlib.pyplot as pltimport pickleprint(f'TensorFlow: {tf.__version__}')print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 2. Configuration

In [ ]:
# Dataset paths — adjust for your environment# Kaggle: /kaggle/input/kermany2018/OCT2017/{train,val,test}# Colab: mount Drive and set appropriate pathsTRAIN_DIR = 'OCT2017/train'VAL_DIR = 'OCT2017/val'TEST_DIR = 'OCT2017/test'INPUT_SHAPE = (224, 224, 3)IMG_SIZE = (224, 224)BATCH_SIZE = 32EPOCHS = 15NUM_CLASSES = 4OUTPUT_KERAS = 'trained_model.keras'OUTPUT_H5 = 'trained_model.h5'HISTORY_PKL = 'training_history.pkl'

## 3. Load Datasets

In [ ]:
training_set = image_dataset_from_directory(    TRAIN_DIR,    labels='inferred',    label_mode='categorical',    color_mode='rgb',    batch_size=BATCH_SIZE,    image_size=IMG_SIZE,    shuffle=True,    interpolation='bilinear',)validation_set = image_dataset_from_directory(    VAL_DIR,    labels='inferred',    label_mode='categorical',    color_mode='rgb',    batch_size=BATCH_SIZE,    image_size=IMG_SIZE,    shuffle=True,    interpolation='bilinear',)print(f'Classes: {training_set.class_names}')

## 4. Build Model (MobileNet V3 + Classification Head)

In [ ]:
# Pretrained MobileNet V3 Large on ImageNet (includes Rescaling preprocess)mobnet = tf.keras.applications.MobileNetV3Large(    input_shape=INPUT_SHAPE,    alpha=1.0,    minimalistic=False,    include_top=True,    weights='imagenet',    classes=1000,    pooling=None,    dropout_rate=0.2,    classifier_activation='softmax',    include_preprocessing=True,)model = tf.keras.models.Sequential([    tf.keras.Input(shape=INPUT_SHAPE),    mobnet,    tf.keras.layers.Dense(units=NUM_CLASSES, activation='softmax'),])model.compile(    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=1e-4),    loss='categorical_crossentropy',    metrics=['accuracy', tf.keras.metrics.F1Score()],)model.summary()

## 5. Train

In [ ]:
training_history = model.fit(    x=training_set,    validation_data=validation_set,    epochs=EPOCHS,)

## 6. Save Model & History

In [ ]:
model.save(OUTPUT_KERAS)model.save(OUTPUT_H5)with open(HISTORY_PKL, 'wb') as f:    pickle.dump(training_history.history, f)print(f'Saved: {OUTPUT_KERAS}, {OUTPUT_H5}, {HISTORY_PKL}')

## 7. Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)plt.figure(figsize=(12, 4))plt.subplot(1, 2, 1)plt.plot(epochs_range, training_history.history['loss'], color='red', label='Training Loss')plt.plot(epochs_range, training_history.history['val_loss'], color='blue', label='Validation Loss')plt.xlabel('Epoch')plt.ylabel('Loss')plt.title('Loss Over Epochs')plt.legend()plt.subplot(1, 2, 2)plt.plot(epochs_range, training_history.history['accuracy'], color='green', label='Training Accuracy')plt.plot(epochs_range, training_history.history['val_accuracy'], color='orange', label='Validation Accuracy')plt.xlabel('Epoch')plt.ylabel('Accuracy')plt.title('Accuracy Over Epochs')plt.legend()plt.tight_layout()plt.show()

## 8. Evaluate on Test Set (Optional)

In [ ]:
if TEST_DIR:    test_set = image_dataset_from_directory(        TEST_DIR,        labels='inferred',        label_mode='categorical',        color_mode='rgb',        batch_size=BATCH_SIZE,        image_size=IMG_SIZE,        shuffle=False,    )    results = model.evaluate(test_set)    print(f'Test loss: {results[0]:.4f}')    print(f'Test accuracy: {results[1]:.4f}')

## 9. Deploy to ViewerAfter training, copy `trained_model.keras` and `trained_model.h5` to the project:```<project-root>/ml_models/eye_disease/trained_model.keras<project-root>/ml_models/eye_disease/trained_model.h5```The Flask viewer loads these automatically on first prediction request.